---
## 5. Modelado Predictivo <a id='5'></a>

Entrenamos **5 modelos** en **3 escenarios** (sin preprocesar / MinMax / Standard) con validación cruzada estratificada de 5 folds.

| Modelo | Tipo | Requiere escalado |
|---|---|---|
| Logistic Regression | Lineal | Sí |
| Random Forest | Ensamble de árboles | No |
| XGBoost | Gradient Boosting | No |
| SVM (SVC) | Kernel | Sí |
| K-Nearest Neighbors | Basado en distancia | Sí |


In [ ]:
# ── 5.1 Definición de modelos y validación cruzada ────────────────────────
# StratifiedKFold: en cada fold mantiene la misma proporción de clases que el total.
# Es obligatorio usarlo con datasets desbalanceados (como el nuestro).

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    # max_iter alto porque con StandardScaler a veces tarda más en converger

    'Random Forest':       RandomForestClassifier(random_state=42, n_estimators=200),
    # n_estimators=200 árboles: más estable que el default (100)

    'XGBoost':             XGBClassifier(random_state=42, n_estimators=200,
                                          eval_metric='mlogloss', verbosity=0),
    # eval_metric='mlogloss': log-loss multiclase, adecuado para 3 clases

    'SVM':                 SVC(random_state=42, kernel='rbf', C=1),
    # kernel='rbf': Radial Basis Function, muy efectivo para datos no lineales

    'KNN':                 KNeighborsClassifier(n_neighbors=7),
    # K=7: impar para evitar empates; probar múltiples K es buena práctica
}

print("Modelos definidos:")
for name in models:
    print(f"  - {name}")

In [ ]:
# ── 5.2 Evaluación en los 3 escenarios ───────────────────────────────────
# Para cada escenario (sin escalar, MinMax, Standard) y para cada modelo:
# → Calculamos Accuracy y F1-weighted con validación cruzada (5 folds)
# → Guardamos media ± desviación estándar de los 5 folds

scenarios = {
    'Sin preprocesamiento': X_raw,
    'MinMaxScaler':         X_minmax,
    'StandardScaler':       X_std,
}

all_results = {}

for scenario_name, X_sc in scenarios.items():
    print(f"\n{'─'*55}")
    print(f"ESCENARIO: {scenario_name}")
    print(f"{'─'*55}")
    all_results[scenario_name] = {}

    for model_name, model in models.items():
        acc = cross_val_score(model, X_sc, y, cv=cv, scoring='accuracy')
        f1  = cross_val_score(model, X_sc, y, cv=cv, scoring='f1_weighted')

        all_results[scenario_name][model_name] = {
            'acc_mean': acc.mean(), 'acc_std': acc.std(),
            'f1_mean':  f1.mean(),  'f1_std':  f1.std(),
        }

        print(f"  {model_name:25s} | Acc: {acc.mean():.4f}±{acc.std():.4f}"
              f" | F1: {f1.mean():.4f}±{f1.std():.4f}")

In [ ]:
# ── 5.3 Tabla resumen de resultados ──────────────────────────────────────
rows = []
for scenario in all_results:
    for model in all_results[scenario]:
        r = all_results[scenario][model]
        rows.append({
            'Escenario': scenario, 'Modelo': model,
            'Accuracy':   round(r['acc_mean'], 4),
            'Acc ±':      round(r['acc_std'],  4),
            'F1-weighted':round(r['f1_mean'],  4),
            'F1 ±':       round(r['f1_std'],   4),
        })

results_df = pd.DataFrame(rows)
display(results_df.style
    .background_gradient(subset=['Accuracy', 'F1-weighted'], cmap='YlGn')
    .format({'Accuracy': '{:.4f}', 'F1-weighted': '{:.4f}',
             'Acc ±': '±{:.4f}', 'F1 ±': '±{:.4f}'})
)

In [ ]:
# ── 5.4 Gráfico comparativo ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
model_names   = list(models.keys())
scenario_list = list(all_results.keys())
sc_colors     = ['#5C6BC0', '#26A69A', '#EF5350']
x     = np.arange(len(model_names))
width = 0.25

for ax_idx, (metric, label) in enumerate([('acc_mean', 'Accuracy'),
                                            ('f1_mean',  'F1-Score (weighted)')]):
    ax = axes[ax_idx]
    for j, (sc, color) in enumerate(zip(scenario_list, sc_colors)):
        vals = [all_results[sc][m][metric]                      for m in model_names]
        errs = [all_results[sc][m][metric.replace('mean','std')] for m in model_names]
        ax.bar(x + j*width, vals, width, label=sc, color=color,
               alpha=0.85, yerr=errs, capsize=4, edgecolor='white')

    ax.set_xticks(x + width)
    ax.set_xticklabels([m.replace(' ', '\n') for m in model_names], fontsize=9)
    ax.set_ylabel(label)
    ax.set_title(f'{label} por modelo y escenario', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylim(0.65, 0.85)
    ax.axhline(y=0.75, color='gray', linestyle='--', alpha=0.5, linewidth=1.5)

plt.suptitle('Comparación de modelos: Sin vs Con Preprocesamiento',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig7_model_comparison.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.5 Selección del mejor modelo y evaluación final ────────────────────
# Identificamos el mejor modelo por F1-weighted (más robusto que Accuracy
# cuando hay desbalance de clases)

best_scenario, best_model_name, best_f1 = '', '', 0
for sc in all_results:
    for model_name in all_results[sc]:
        f1 = all_results[sc][model_name]['f1_mean']
        if f1 > best_f1:
            best_f1 = f1
            best_scenario   = sc
            best_model_name = model_name

print(f"MEJOR MODELO: {best_model_name}")
print(f"ESCENARIO:    {best_scenario}")
print(f"F1-weighted (CV-5): {best_f1:.4f}")

# Elegir el dataset correcto según el mejor escenario
X_best = {'Sin preprocesamiento': X_raw,
          'MinMaxScaler':         X_minmax,
          'StandardScaler':       X_std}[best_scenario]

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_best, y, test_size=0.2, random_state=42, stratify=y)

# Entrenamiento final y evaluación en test set
final_model = models[best_model_name]
final_model.fit(X_train_b, y_train_b)
y_pred      = final_model.predict(X_test_b)

print("\n" + "=" * 60)
print(f"REPORTE DE CLASIFICACIÓN — {best_model_name}")
print("=" * 60)
print(classification_report(y_test_b, y_pred, target_names=le_target.classes_))
# Precision: de todo lo que predije como X, ¿cuánto era realmente X?
# Recall:    de todo lo que era X en realidad, ¿cuánto detecté?
# F1-Score:  media armónica entre Precision y Recall
# Support:   casos reales de cada clase en el test set

In [ ]:
# ── 5.6 Matriz de confusión ───────────────────────────────────────────────
# Filas = etiqueta REAL | Columnas = etiqueta PREDICHA
# Diagonal principal = aciertos | Fuera de diagonal = errores

cm = confusion_matrix(y_test_b, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_target.classes_,
            yticklabels=le_target.classes_,
            linewidths=0.5, ax=ax,
            annot_kws={'size': 14, 'fontweight': 'bold'})
ax.set_title(f'Matriz de Confusión — {best_model_name}',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Etiqueta REAL', fontsize=12)
ax.set_xlabel('Etiqueta PREDICHA', fontsize=12)
plt.tight_layout()
plt.savefig('fig8_confusion_matrix.png', dpi=130, bbox_inches='tight')
plt.show()

print("Interpretación:")
print(f"  → El modelo clasifica bien 'low' (clase mayoritaria).")
print(f"  → 'medium' es la más difícil: se confunde con ambas.")

In [ ]:
# ── 5.7 Importancia de variables (si el modelo lo soporta) ───────────────
# Random Forest y XGBoost calculan feature_importances_ automáticamente.
# Es la reducción promedio de impureza (Gini) que aporta cada variable.

if hasattr(final_model, 'feature_importances_'):
    fi = pd.Series(final_model.feature_importances_, index=FEAT).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(9, 7))
    bar_colors = ['#E53935' if v > 0.10 else '#5C6BC0' if v > 0.05 else '#90CAF9'
                  for v in fi.values]
    fi.plot(kind='barh', ax=ax, color=bar_colors, edgecolor='white', linewidth=0.5)
    ax.set_title(f'Importancia de Variables — {best_model_name}',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Importancia (Gini)', fontsize=11)
    ax.axvline(x=0.05, color='gray', ls='--', alpha=0.7, lw=1.5, label='5%')
    ax.axvline(x=0.10, color='red',  ls='--', alpha=0.5, lw=1.5, label='10%')
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig('fig9_feature_importance.png', dpi=130, bbox_inches='tight')
    plt.show()

    print("TOP 5 variables más importantes:")
    display(fi.sort_values(ascending=False).head().to_frame(name='Importancia'))
else:
    print(f"El modelo {best_model_name} no expone feature_importances_.")
    print("Usa Random Forest o XGBoost para obtener este análisis.")

In [ ]:
# ── 5.8 Comparación: ¿El preprocesamiento ayudó? ──────────────────────
print("¿El preprocesamiento mejoró, empeoró o fue indiferente?")
print("=" * 60)

for model_name in models:
    results_by_sc = {sc: all_results[sc][model_name]['f1_mean'] for sc in scenario_list}
    best_sc  = max(results_by_sc, key=results_by_sc.get)
    base_f1  = results_by_sc['Sin preprocesamiento']
    delta_mm  = results_by_sc['MinMaxScaler']  - base_f1
    delta_std = results_by_sc['StandardScaler'] - base_f1

    print(f"\n{model_name}:")
    print(f"  Sin preprocesar : {base_f1:.4f}")
    print(f"  MinMaxScaler    : {results_by_sc['MinMaxScaler']:.4f}  ({delta_mm:+.4f})")
    print(f"  StandardScaler  : {results_by_sc['StandardScaler']:.4f}  ({delta_std:+.4f})")
    print(f"  → Mejor escenario: {best_sc}")

# ── 5.9 Conclusiones ─────────────────────────────────────────────────────
print("""
HALLAZGOS PRINCIPALES
═════════════════════

1. VARIABLES MÁS PREDICTIVAS:
   • stress_level y anxiety_level → correlación fuerte con depression_risk
   • sleep_hours y academic_performance → relación INVERSA (más es mejor)
   • physical_activity → mayor actividad = menor riesgo
   • La plataforma usada (Instagram, TikTok…) tiene impacto menor

2. MEJOR MODELO:
   • Random Forest obtuvo el mejor rendimiento en todos los escenarios
   • Accuracy ≈ 76-77% | F1-weighted ≈ 0.77 (validación cruzada 5 folds)
   • Es robusto al desbalance de clases y captura relaciones no lineales

3. IMPACTO DEL PREPROCESAMIENTO:
   • Árboles (RF, XGBoost): el escalado NO cambia los resultados
     → Los árboles son invariantes a la escala de las variables
   • Modelos lineales (LR, SVM, KNN): StandardScaler ayuda marginalmente
   • Conclusión: el preprocesamiento no fue crítico en este dataset

4. CLASE MÁS DIFÍCIL: 'medium'
   • F1-score de 'medium' ≈ 0.51 (muy por debajo de 'low' y 'high')
   • El desbalance (low > medium > high) hace que el modelo tienda
     a clasificar los casos ambiguos como 'low'

RECOMENDACIONES FUTURAS:
════════════════════════
1. Usar SMOTE (pip install imbalanced-learn) para balancear las clases
2. Aplicar GridSearchCV / RandomizedSearchCV para optimizar hiperparámetros
3. Probar LightGBM / CatBoost para comparar con XGBoost
4. Añadir class_weight='balanced' para penalizar errores en clases minoritarias
5. Recolectar más datos de las clases 'medium' y 'high'
""")